In [0]:
from pyspark.sql.functions import *

### Fetch Job Parameter

In [0]:
bronze_root_path = dbutils.widgets.get("bronze_root_path")
bronze_table_name = dbutils.widgets.get("table_name")
file_formart = dbutils.widgets.get("csv_file_formart")

### Define Paths

In [0]:
landing_path = f"{bronze_root_path}/{table_name}/year(current_date())/month(current_date())/day(current_date())"
schema_location = f"{bronze_root_path}/{table_name}/schema/"
checkpoint_location = f"{bronze_root_path}/{table_name}/checkpoint/"

### Read Files Using AutoLoader

In [0]:
df = (
    spark.readStream.format("cloudFiles")
                    .option("cloudFiles.format", file_format)
                    .option("cloudFiles.useManagedFileEvents", "true")
                    .option("PathGlobFilter", file_format)
                    .option("cloudFiles.schemaLocation", schema_location)
                    .option("header", "true")
                    .load(landing_path)
)

print("The schema of the read file is :  ")
df.printSchema()

### Write the DataFrame into Bronze Table

In [0]:
(
    df.writeStream.format("delta")
                  .option("checkpointLocation", checkpoint_location)
                  .trigger(availableNow = True)
                  .toTable(bronze_table_name)
)